# Regressão: aluguel em Munique (Rent99)

Queremos prever o aluguel mensal em euros (`rent`) a partir das características do imóvel. O notebook tem duas partes:

1. **Modelos lineares** (linear, ridge, lasso e polinomial), comparados por LOOCV, com análise de resíduos, coeficientes e intervalos de confiança.
2. **Árvores** (árvore com e sem poda, bagging e random forest), com hiperparâmetros escolhidos por `GridSearch` e K-Fold.

As duas partes usam a mesma divisão treino/teste (`split_rent99`, 2465 imóveis de treino e 617 de teste), e no fim todos os modelos são comparados no mesmo teste.

In [ ]:
import time

import altair as alt
import numpy as np
import pandas as pd
from IPython.display import display

from rbg.Regression.Linear import Linear
from rbg.Regression.Ridge import Ridge
from rbg.Regression.Lasso import Lasso
from rbg.Regression.Polynomial import Polynomial
from rbg.DecisionTree import RegressionTree, BaggingReg, RandomForestRegression
from rbg.ModelSelection import GridSearch, KFold, LOOCV
from rbg.Utilities import MSE, RMSE, RSquare

from utils.dados import SEED, carregar_rent99, split_rent99, salvar_resultados
from utils.avaliacao import metricas_regressao
from utils.graficos import configurar_altair, plot_arvore

configurar_altair()

## Parte 1: modelos lineares

### Observações iniciais

Antes de escolher as variáveis, olhamos de novo para a relação de cada uma com o aluguel, incluindo duas transformações da área (log e inverso) e o ano de construção agrupado por década.

In [ ]:
dados_reg = carregar_rent99()
display(dados_reg.head())

grafico_area_aluguel = alt.Chart(dados_reg).mark_circle(opacity=0.5).encode(
    x=alt.X("area:Q", title="Área"),
    y=alt.Y("rent:Q", title="Aluguel")
).properties(width=300, height=250)

dados_reg["decada"] = (dados_reg["yearc"] // 10 * 10).astype(int)

grafico_decada_aluguel = alt.Chart(dados_reg).mark_boxplot().encode(
    x=alt.X("decada:O", title="Década de construção"),
    y=alt.Y("rent:Q", title="Aluguel")
).properties(width=400, height=250)
dados_reg["log(area)"] = np.log(dados_reg["area"])
dados_reg["inv(area)"] = 1 / dados_reg["area"]

grafico_area_log_aluguel = alt.Chart(dados_reg).mark_circle(opacity=0.5).encode(
    x=alt.X("log(area):Q", title="Área (log)"),
    y=alt.Y("rent:Q", title="Aluguel")
).properties(width=300, height=250)

grafico_area_inversa_aluguel = alt.Chart(dados_reg).mark_circle(opacity=0.5).encode(
    x=alt.X("inv(area):Q", title="Área (inverso)"),
    y=alt.Y("rent:Q", title="Aluguel")
).properties(width=300, height=250)

graficos = [grafico_area_aluguel, grafico_area_log_aluguel, grafico_area_inversa_aluguel, grafico_decada_aluguel]

for coluna in ['location', 'bath', 'kitchen', 'cheating']:
    graficos.append(alt.Chart(dados_reg).mark_boxplot().encode(
        x=alt.X(f'{coluna}:N', title=coluna),
        y=alt.Y('rent:Q', title='Aluguel')
    ).properties(title=f'Aluguel por {coluna}'))

alt.vconcat(alt.hconcat(*graficos[:4]), alt.hconcat(*graficos[4:]))

Nos gráficos, a relação entre ano de construção e aluguel parece menos evidente. A área na escala original parece se adequar a um ajuste linear. Vou comparar os tipos de ajuste pela validação.

### 1.1 Escolha das variáveis

Vou manter `area` e `yearc` na escala original e `bath`, `kitchen` e `cheating` como 0 ou 1.

Para `location`, vou criar `l2` (1 se location = 2) e `l3` (1 se location = 3), com 0 nos demais casos. A referência será location = 1.

Vou deixar `district`, `decada`, `log(area)` e `inv(area)` de fora nesta comparação.

In [ ]:
TARGET_REG = 'rent'
FEATURES_REG = ['area', 'yearc', 'bath', 'kitchen', 'cheating', 'l2', 'l3']
dados_reg["l2"] = (dados_reg["location"] == 2).astype(int)
dados_reg["l3"] = (dados_reg["location"] == 3).astype(int)

dados_reg[["location", "l2", "l3"]].head()

### 1.2 Separar o teste final

Vou reservar 20% para o teste final. Os 80% restantes serão usados para escolher os modelos com LOOCV.

In [ ]:
X_dev, X_teste, y_dev, y_teste = split_rent99(dados_reg[FEATURES_REG], dados_reg[TARGET_REG])

### 1.3 Conferir os dados antes dos ajustes

Conferimos valores ausentes, valores inválidos e as dimensões dos arrays.

In [ ]:
# Conferir se há valores ausentes nos dados de treino.
display(pd.DataFrame(X_dev, columns=FEATURES_REG).isna().sum())

# Os arrays precisam ter valores numéricos finitos e dimensões compatíveis.
assert np.isfinite(X_dev).all()
assert np.isfinite(y_dev).all()
assert np.isfinite(X_teste).all()
assert np.isfinite(y_teste).all()
assert X_dev.shape[0] == len(y_dev)
assert X_teste.shape[0] == len(y_teste)
assert X_dev.shape[1] == X_teste.shape[1] == len(FEATURES_REG)

print('Treino:', X_dev.shape, y_dev.shape)
print('Teste:', X_teste.shape, y_teste.shape)

### 1.4 Preparar o LOOCV

Vou usar as mesmas rodadas de LOOCV para comparar todos os modelos.

In [ ]:
# Cada rodada retorna os índices de treino e depois os de validação.
rodadas = list(LOOCV().split(X_dev))

### 1.5 Ajustar os modelos

Vou testar vários modelos com LOOCV para escolher o mais adequado, variando os valores de `alpha` de Ridge e Lasso e alguns graus polinomiais.

In [ ]:
modelos_reg = {'linear': (Linear, {})}

for i in range(0, 8):
    alpha = 2 ** (4 - i)
    modelos_reg[f"ridge_2^{4-i}"] = (Ridge, {"alpha": alpha})
    modelos_reg[f"lasso_2^{4-i}"] = (Lasso, {"alpha": alpha})

for grau in [2, 3]:
    modelos_reg[f"polinomial_{grau}"] = (
        Polynomial,
        {"degrees": [grau] * X_dev.shape[1]}
    )

inicio = time.time()
resultados_validacao = []

for nome, (classe, parametros) in modelos_reg.items():
    previsoes_loocv = np.empty(len(y_dev))

    for idx_treino, idx_val in rodadas:
        modelo = classe(**parametros)
        modelo.fit(X_dev[idx_treino], y_dev[idx_treino])
        previsoes_loocv[idx_val] = modelo.predict(X_dev[idx_val])

    resultados_validacao.append({
        'modelo': nome,
        'RMSE': RMSE(previsoes_loocv, y_dev),
        'R²': RSquare(previsoes_loocv, y_dev),
    })

print(f"LOOCV de {len(modelos_reg)} modelos em {time.time() - inicio:.0f} s")

### 1.6 Comparar os resultados

In [ ]:
tabela_reg = pd.DataFrame(resultados_validacao).sort_values('RMSE').reset_index(drop=True)
display(tabela_reg.round(4))
melhor_modelo = 'linear'  # Escolha pela simplicidade e interpretabilidade.

Ridge, lasso e linear costumam ficar muito próximos no LOOCV com estas sete variáveis, porque há muito mais imóveis do que variáveis e a regularização quase não muda os coeficientes. Os polinomiais podem ter RMSE um pouco menor, o que indica um ganho na validação e não um sobreajuste. Mesmo assim, preferimos o modelo linear pela simplicidade e interpretabilidade, e os outros entram na comparação final no teste.

### 1.7 Avaliação final

Vou treinar o modelo escolhido em todos os dados de treino e avaliar seu desempenho no teste com RMSE, MAE, R² e gráficos de resíduos.

In [ ]:
modelo_final = Linear()

modelo_final.fit(X_dev, y_dev)
previsoes_teste = modelo_final.predict(X_teste)

print(f"Já com o modelo linear escolhido, RMSE: {RMSE(previsoes_teste, y_teste):.4f}, R²: {RSquare(previsoes_teste, y_teste):.4f}")

inferencia = modelo_final.inference()

#### 1.7.1 Desempenho no teste

Vou avaliar o desempenho do modelo linear. No gráfico, pontos próximos da diagonal indicam previsões próximas do aluguel real.

In [ ]:
beta, sigma2, cov_beta, residuos_dev = inferencia
previsoes_dev = modelo_final.predict(X_dev)

metricas_finais = pd.DataFrame([
    metricas_regressao('Linear (treino)', previsoes_dev, y_dev),
    metricas_regressao('Linear (teste)', previsoes_teste, y_teste),
])
display(metricas_finais.round(4))

Se o RMSE do teste ficar próximo do RMSE do treino, o modelo não está sobreajustado. O gráfico abaixo mostra onde ele erra mais: nos aluguéis muito altos, que são previstos abaixo do valor real.

In [ ]:
dados_teste = pd.DataFrame({'real': y_teste, 'previsto': previsoes_teste})
limites = [min(y_teste.min(), previsoes_teste.min()),
           max(y_teste.max(), previsoes_teste.max())]
diagonal = pd.DataFrame({'valor': limites})

grafico_real_previsto = alt.Chart(dados_teste).mark_circle(opacity=0.4).encode(
    x=alt.X('real:Q', title='Aluguel real (€)', scale=alt.Scale(domain=limites)),
    y=alt.Y('previsto:Q', title='Aluguel previsto (€)', scale=alt.Scale(domain=limites))
)
grafico_linha_ideal = alt.Chart(diagonal).mark_line(color='red').encode(
    x='valor:Q', y='valor:Q'
)
(grafico_real_previsto + grafico_linha_ideal).properties(width=350, height=300)

#### 1.7.2 Resíduos

Vou usar os resíduos retornados por `modelo_final.inference()`, que correspondem ao treino. Espero pontos espalhados em torno de zero, sem curvatura ou formato de funil. O histograma ajuda a observar assimetria e erros extremos.

In [ ]:
dados_residuos = pd.DataFrame({
    'previsto': previsoes_dev,
    'residuo': residuos_dev
})

grafico_residuos_previstos = alt.Chart(dados_residuos).mark_circle(opacity=0.3).encode(
    x=alt.X('previsto:Q', title='Aluguel previsto (€)'),
    y=alt.Y('residuo:Q', title='Resíduo (€)')
)
grafico_linha_zero = alt.Chart(pd.DataFrame({'zero': [0]})).mark_rule(color='red').encode(
    y='zero:Q'
)
grafico_histograma_residuos = alt.Chart(dados_residuos).mark_bar().encode(
    x=alt.X('residuo:Q', bin=alt.Bin(maxbins=25), title='Resíduo (€)'),
    y=alt.Y('count():Q', title='Quantidade de imóveis')
)
alt.hconcat(
    (grafico_residuos_previstos + grafico_linha_zero).properties(width=350, height=280),
    grafico_histograma_residuos.properties(width=350, height=280),
)

O formato de funil nos resíduos é preocupante: sugere que a variância dos erros aumenta com o aluguel previsto, contrariando a hipótese de variância constante. O histograma não permite confirmar a normalidade, mas seu formato me parece razoável.

#### 1.7.3 Betas na escala original e padronizada

Vou exibir os coeficientes originais e os padronizados por β × desvio da variável / desvio do aluguel.

In [ ]:
coeficientes = beta[1:]
coeficientes_padronizados = coeficientes * X_dev.std(axis=0) / y_dev.std()

comparacao_coeficientes = pd.DataFrame({
    'variavel': FEATURES_REG,
    'beta_original': coeficientes,
    'beta_padronizado': coeficientes_padronizados
})
display(comparacao_coeficientes.round(4))

O `beta_original` indica quanto o valor previsto muda com o aumento de uma unidade da variável contínua, mantendo as outras fixas. Nas variáveis binárias, indica a diferença prevista entre os casos 1 e 0. Para `l2` e `l3`, a comparação é com `location = 1`.

Já o beta padronizado permite comparar o tamanho das associações em uma mesma escala. Aqui, a área tem o maior valor, seguida pelo ano, `cheating`, `l2` e `l3`. Os valores de `kitchen` e `bath` são menores.

#### 1.7.4 Coeficientes

Aqui vou usar `cov_beta` para ter uma ideia da incerteza de cada beta. A raiz da diagonal dá os erros padrão. Com eles, vou calcular intervalos de 95% aproximados usando beta ± 1,96 × erro padrão, mantendo a escala original.

Esses intervalos pressupõem erros não correlacionados e variância constante. Como os resíduos mostraram um formato de funil, vou interpretar os intervalos com cuidado.

In [ ]:
erros_padrao = np.sqrt(np.diag(cov_beta)[1:])

tabela_coeficientes = pd.DataFrame({
    'variavel': FEATURES_REG,
    'coeficiente': coeficientes,
    'erro_padrao': erros_padrao,
    'inferior': coeficientes - 1.96 * erros_padrao,
    'superior': coeficientes + 1.96 * erros_padrao
})
display(tabela_coeficientes.round(4))

Se um intervalo não contém o zero, a variável tem associação com o aluguel mesmo depois de controlar pelas outras. Como os resíduos mostram variância crescente, os erros padrão tendem a ser otimistas e os intervalos verdadeiros devem ser um pouco mais largos.

## Parte 2: árvores de regressão

Faremos agora o uso de árvores para prever os nossos dados. Usamos a mesma separação entre treino e teste da parte 1. Em seguida, aplicaremos aos nossos dados as árvores de regressão com e sem poda, bagging e Random Forest. Começaremos com os valores padrão, a fim de ilustrar o uso da biblioteca, e faremos um gráfico comparativo entre a árvore com poda e sem poda. Além do erro calculado com o MSE, também calcularemos o erro OOB do modelo de bagging ou random forest escolhido.

* **Validação**: para árvores, random forest e bagging, usaremos o GridSearch para procurar, em cada um dos três, os melhores hiperparâmetros.
* **Comparação de modelos**: com os modelos ajustados pelos seus melhores hiperparâmetros, faremos uma validação cruzada com os dados de treino para decidir qual o melhor modelo.
* **Avaliação final**: dado o modelo escolhido, faremos a predição dos dados de teste e calcularemos seu erro.
* **Interpretação dos resultados**: considerações sobre a escolha das métricas e o trade-off entre viés e variância.

Diferente da parte 1, aqui usamos `location` e `district` como estão: a árvore não precisa de variáveis indicadoras, porque ela só faz cortes do tipo "variável ≤ valor".

In [ ]:
colunas_X = ['area', 'yearc', 'location', 'bath', 'kitchen', 'cheating', 'district']

X_treino, X_teste_arv, y_treino, y_teste_arv = split_rent99(dados_reg[colunas_X], dados_reg['rent'])

# mesma divisão da parte 1: os imóveis de teste são os mesmos
assert np.array_equal(y_teste_arv, y_teste)
print(X_treino.shape, X_teste_arv.shape)

In [ ]:
arvore = RegressionTree()

arvore.fit(X_treino, y_treino)

y_previsto = arvore.predict(X_teste_arv)

erro_arvore_sem_poda = MSE(y_previsto, y_teste)

print("erro (MSE) da árvore sem poda:", erro_arvore_sem_poda)

In [ ]:
inicio = time.time()
arvore_poda = RegressionTree()

arvore_poda.fit(X_treino, y_treino, prune=True)

y_previsto_poda = arvore_poda.predict(X_teste_arv)

erro_arvore_com_poda = MSE(y_previsto_poda, y_teste)

print("erro (MSE) da árvore com poda:", erro_arvore_com_poda)
print(f"tempo do ajuste com poda: {time.time() - inicio:.0f} s")

In [ ]:
arvorebag = BaggingReg()

arvorebag.fit(X_treino, y_treino)

y_previstobag = arvorebag.predict(X_teste_arv)

erro_arvore_bag = MSE(y_previstobag, y_teste)

print("erro (MSE) do bagging:", erro_arvore_bag)

In [ ]:
arvore_random = RandomForestRegression()

arvore_random.fit(X_treino, y_treino)

y_previsto_random = arvore_random.predict(X_teste_arv)

erro_arvore_random = MSE(y_previsto_random, y_teste)

print("erro (MSE) do random forest:", erro_arvore_random)

Plot da árvore com poda:

In [ ]:
preditores = colunas_X

plot_arvore(arvore_poda, feature_names=preditores)

Plot da árvore sem poda (a figura é limitada em largura, então os rótulos se sobrepõem; o que importa aqui é o tamanho da árvore):

In [ ]:
n_folhas = int(np.sum(arvore.get_tree_structure_info()['children_left'] == -1))
n_folhas_poda = int(np.sum(arvore_poda.get_tree_structure_info()['children_left'] == -1))
print(f"folhas da árvore sem poda: {n_folhas}; com poda: {n_folhas_poda}")

plot_arvore(arvore, feature_names=preditores)

A árvore sem poda cresce até quase uma folha por imóvel, o que é impossível de ler e tende a sobreajustar. A poda corta os ramos que pouco reduzem o erro e deixa uma árvore muito menor. Se o que mais desejamos é interpretabilidade, a árvore com poda é a melhor opção entre as árvores; o custo é o tempo do ajuste, porque a poda testa várias subárvores.

## Validação

### Validação da árvore

Uma vez que árvores (até mesmo as mais simples, sem poda e sem bagging) têm um custo grande para serem montadas de acordo com o algoritmo implementado (a escolha do melhor corte passa por todos os valores, etc.), e uma vez que as árvores possuem vários parâmetros, então para fazer uma validação otimizada usaremos apenas os hiperparâmetros:

- min_samples_split (e não usaremos min_samples_leaf, que ficará padronizado como 1)
- max_depth (e não usaremos max_leaf)
- min_impurity_decrease
- criterion_split (e o nosso "criterion_evaluation" será portanto igual ao criterion_split)
- criterion_choose

Dentro de criterion_choose, também iremos apenas avaliar a média e a mediana, já que preços contínuos não costumam ter moda, e também deixaremos de fora "min" e "max", pois em problemas de estimar preço o mais natural é usar a mediana ou a média, já que "min" e "max" são justamente valores de extremo.

Também nos limitaremos a apenas MSE, RMSE e MAE para serem os critérios de cálculo de impureza ("Standard" é o MSE).

Em todas as validações usamos o K-Fold com 5 partes, embaralhado com a semente do projeto.

In [ ]:
p_grid = {
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 10, 50],
    'min_impurity_decrease': [0.01, 0.1, 1, 10],
    'criterion_split': ['Standard', 'RMSE', 'MAE'],
    'criterion_choose': ['Standard', 'median']
}

grid_arvore = GridSearch(model=RegressionTree(), param_grid=p_grid, metrics=MSE,
                         cv=KFold(k=5, shuffle=True, random_state=SEED))

In [ ]:
inicio = time.time()
grid_arvore.fit(X_treino, y_treino)
print(f"GridSearch da árvore: {time.time() - inicio:.0f} s")

Agora iremos mostrar os parâmetros escolhidos para a nossa árvore.

In [ ]:
print(grid_arvore.best_params_)
print("RMSE médio na validação:", np.sqrt(grid_arvore.best_score_))

### Validação do bagging

As árvores de bagging foram implementadas apenas com NumPy (sem paralelização), então as árvores são construídas uma de cada vez e um único bagging já é demorado. Portanto, os modelos que iremos testar serão menores ainda: para o bagging mudaremos também a quantidade de árvores utilizadas, e faremos a busca dos hiperparâmetros vistos anteriormente com menos valores, para ser viável computacionalmente.

In [ ]:
pbaggin_grid = {
    'max_depth': [10],
    'min_samples_split': [10],
    'min_impurity_decrease': [1, 10],
    'criterion_split': ['Standard', 'RMSE'],
    'criterion_choose': ['Standard', 'median'],
    'n_trees': [10, 20]
}

grid_bagging = GridSearch(model=BaggingReg(), param_grid=pbaggin_grid, metrics=MSE,
                          cv=KFold(k=5, shuffle=True, random_state=SEED))

In [ ]:
inicio = time.time()
grid_bagging.fit(X_treino, y_treino)
print(f"GridSearch do bagging: {time.time() - inicio:.0f} s")
print(grid_bagging.best_params_)
print("RMSE médio na validação:", np.sqrt(grid_bagging.best_score_))

### Validação do random forest

Similarmente, para o random forest vamos testar apenas 10 e 20 árvores, com o número de preditores sorteados em cada corte igual ao piso da raiz quadrada do número total de preditores. Como o nosso dataset tem 7 preditores, sempre dois preditores serão sorteados em cada corte.

In [ ]:
p_random_grid = {
    'max_depth': [10],
    'min_samples_split': [10],
    'min_impurity_decrease': [1, 10],
    'criterion_split': ['Standard', 'RMSE'],
    'criterion_choose': ['Standard', 'median'],
    'n_trees': [10, 20],
    'factor_selection': [np.sqrt]
}
grid_RandomReg = GridSearch(model=RandomForestRegression(), param_grid=p_random_grid, metrics=MSE,
                            cv=KFold(k=5, shuffle=True, random_state=SEED))

In [ ]:
inicio = time.time()
grid_RandomReg.fit(X_treino, y_treino)
print(f"GridSearch do random forest: {time.time() - inicio:.0f} s")
print(grid_RandomReg.best_params_)
print("RMSE médio na validação:", np.sqrt(grid_RandomReg.best_score_))

## Seleção de modelo

Para a seleção de modelos, faremos um K-Fold usando os campeões de cada GridSearch: treinamos cada um nas partes de treino e escolhemos aquele que teve o menor erro médio (MSE) nas partes de validação. Usamos uma semente diferente da do GridSearch para que as partes não sejam as mesmas que escolheram os hiperparâmetros.

In [ ]:
param_r = grid_RandomReg.best_params_
param_bagging = grid_bagging.best_params_
param_arvore = grid_arvore.best_params_

baggingReg = BaggingReg(**param_bagging)
arvore_pos_grid = RegressionTree(**param_arvore)
randomReg = RandomForestRegression(**param_r)

In [ ]:
k_fold = KFold(k=5, shuffle=True, random_state=SEED + 1)
lista_erros_arvore = []
lista_erros_bag = []
lista_erros_random = []

for treino_k, teste_k in k_fold.split(X_treino):
    X_treino_k, X_teste_k = X_treino[treino_k], X_treino[teste_k]
    y_treino_k, y_teste_k = y_treino[treino_k], y_treino[teste_k]

    # árvore nova a cada parte: refazer o fit no mesmo objeto de árvore da rbg
    # reaproveita estado do ajuste anterior e piora a árvore
    arvore_pos_grid = RegressionTree(**param_arvore)
    randomReg.fit(X_treino_k, y_treino_k)
    baggingReg.fit(X_treino_k, y_treino_k)
    arvore_pos_grid.fit(X_treino_k, y_treino_k)
    lista_erros_arvore.append(MSE(arvore_pos_grid.predict(X_teste_k), y_teste_k))
    lista_erros_bag.append(MSE(baggingReg.predict(X_teste_k), y_teste_k))
    lista_erros_random.append(MSE(randomReg.predict(X_teste_k), y_teste_k))

erros_kfold = pd.DataFrame({
    'modelo': ['árvore', 'bagging', 'random forest'],
    'MSE médio': [np.mean(lista_erros_arvore), np.mean(lista_erros_bag), np.mean(lista_erros_random)],
})
erros_kfold['RMSE médio'] = np.sqrt(erros_kfold['MSE médio'])
erros_kfold.round(1)

## Avaliação final

Com o nosso modelo ganhador, treinamos em todos os dados de treino, aplicamos aos dados de teste e calculamos o erro.

In [ ]:
arvore_pos_grid = RegressionTree(**param_arvore)  # objeto novo para o ajuste final
modelos_arvore = {'árvore': arvore_pos_grid, 'bagging': baggingReg, 'random forest': randomReg}
vencedor = erros_kfold.sort_values('MSE médio').iloc[0]['modelo']
print("vencedor:", vencedor)

# ajusta os três no treino completo: o vencedor é o modelo final, os outros entram na tabela comparativa
previsoes_arvores = {}
for nome, modelo in modelos_arvore.items():
    modelo.fit(X_treino, y_treino)
    previsoes_arvores[nome] = modelo.predict(X_teste_arv)

predicao_final = previsoes_arvores[vencedor]
print("Erro (MSE) do modelo no teste:", MSE(predicao_final, y_teste))
print("RMSE no teste:", RMSE(predicao_final, y_teste))

Como bagging e random forest guardam quais imóveis ficaram fora de cada amostra bootstrap, podemos calcular o erro OOB (out-of-bag) sem usar o teste. O OOB usa o critério de avaliação da árvore, que aqui é o mesmo do corte.

In [ ]:
ensemble = baggingReg if vencedor != 'random forest' else randomReg
nome_ensemble = 'bagging' if vencedor != 'random forest' else 'random forest'
oob = ensemble.OOB()
print(f"OOB do {nome_ensemble} (critério {ensemble.criterion_split}):", oob)

## Interpretação dos resultados

Bagging e random forest tendem a vencer a árvore única porque fazem a média de várias árvores, o que diminui a variância. Árvores sem poda têm viés pequeno (podem se ajustar a quase qualquer padrão do treino), mas variância grande; a média de várias árvores reduz essa variância sem aumentar muito o viés, que é o trade-off entre viés e variância.

O random forest só deveria ganhar do bagging com muitas árvores: sorteando variáveis em cada corte ele deixa as árvores menos parecidas, mas cada árvore fica pior. Com apenas 10 ou 20 árvores (limite imposto pelo custo computacional) e só 7 preditores, dos quais apenas 2 são sorteados em cada corte, essa vantagem pode não aparecer. Com paralelização e mais árvores, a comparação poderia mudar.

Tanto o GridSearch quanto os modelos de árvore permitem outras métricas, que poderiam ser exploradas a depender dos critérios de decisão.

Um adendo é que poderia ter sido aplicada uma poda na árvore campeã do GridSearch. Como a poda foi planejada para ser um parâmetro do `fit`, ela não entrou como um dos parâmetros do grid; abaixo comparamos, no mesmo K-Fold da seleção, a árvore campeã com poda e o bagging.

In [ ]:
inicio = time.time()
lista_erros_poda = []
for treino_k, teste_k in k_fold.split(X_treino):
    arvore_k = RegressionTree(**param_arvore)
    arvore_k.fit(X_treino[treino_k], y_treino[treino_k], prune=True)
    lista_erros_poda.append(MSE(arvore_k.predict(X_treino[teste_k]), y_treino[teste_k]))

print("RMSE médio da árvore campeã com poda:", np.sqrt(np.mean(lista_erros_poda)))
print("RMSE médio do bagging:", np.sqrt(np.mean(lista_erros_bag)))
print(f"tempo: {time.time() - inicio:.0f} s")

A árvore sem poda cresce até quase uma folha por imóvel, o que é impossível de ler e tende a sobreajustar. A poda corta os ramos que pouco reduzem o erro e deixa uma árvore muito menor. Se o que mais desejamos é interpretabilidade, a árvore com poda é a melhor opção entre as árvores; o custo é o tempo do ajuste, porque a poda testa várias subárvores._CAMPEA

## Comparação final no teste

Para fechar, ajustamos no treino completo cada família de modelo com os hiperparâmetros escolhidos na validação e medimos todos no mesmo teste de 617 imóveis. Ridge e lasso entram com o `alpha` que teve o menor RMSE no LOOCV. A escolha de cada modelo foi feita só com o treino; o teste serve apenas para comparar.

In [ ]:
def melhor_alpha(prefixo):
    linhas = tabela_reg[tabela_reg['modelo'].str.startswith(prefixo)]
    nome = linhas.sort_values('RMSE').iloc[0]['modelo']
    return modelos_reg[nome][1]['alpha'], nome

alpha_ridge, nome_ridge = melhor_alpha('ridge')
alpha_lasso, nome_lasso = melhor_alpha('lasso')

lineares = {
    'Linear': Linear(),
    f'Ridge (alpha = {alpha_ridge:g})': Ridge(alpha=alpha_ridge),
    f'Lasso (alpha = {alpha_lasso:g})': Lasso(alpha=alpha_lasso),
    'Polinomial (grau 2)': Polynomial(degrees=[2] * X_dev.shape[1]),
}

comparacao = []
for nome, modelo in lineares.items():
    modelo.fit(X_dev, y_dev)
    comparacao.append(metricas_regressao(nome, modelo.predict(X_teste), y_teste))

comparacao.append(metricas_regressao('Árvore com poda (padrão)', y_previsto_poda, y_teste))
comparacao.append(metricas_regressao('Árvore (GridSearch)', previsoes_arvores['árvore'], y_teste))
comparacao.append(metricas_regressao(f"Bagging ({param_bagging['n_trees']} árvores)", previsoes_arvores['bagging'], y_teste))
comparacao.append(metricas_regressao(f"Random forest ({param_r['n_trees']} árvores)", previsoes_arvores['random forest'], y_teste))

tabela_teste = pd.DataFrame(comparacao).sort_values('RMSE').reset_index(drop=True)
tabela_teste.round(3)

In [ ]:
salvar_resultados("regressao", tabela_teste)

## Fechamento

A tabela acima reúne todos os modelos de regressão no mesmo teste; a rede neural (notebook 05) entra na comparação do relatório final. Nos dois tipos de modelo, o erro é dominado pelos aluguéis mais altos, que são raros e muito dispersos. Entre modelos com desempenho parecido, o linear é o mais fácil de explicar: cada coeficiente diz quanto o aluguel muda com cada característica, e ele treina em uma fração de segundo, enquanto os conjuntos de árvores precisam de busca de hiperparâmetros e muito mais tempo.